In [6]:
import os
import numpy as np
import h5py
from tqdm import tqdm
import re

In [7]:
ROOT = "/restricteddata/ukaea/gyrokinetics"

In [8]:
def K_files(directory):
    files = os.listdir(directory)
    digit_files = sorted(
        [file for file in files if file.isdigit()], key=lambda x: int(x)
    )
    k_files = sorted(
        [file for file in files if file.startswith("K") and not file.endswith(".dat")]
    )
    return k_files + digit_files


def poten_files(directory):
    files = os.listdir(directory)
    poten_files = sorted([file for file in files if file.startswith("Poten")])
    timestep_slices = [int(f.replace("Poten", "")) for f in poten_files]
    return poten_files, np.array(timestep_slices) - 1

In [9]:
def parse_input_dat(file_path):
    parsed_data = {}
    with open(file_path, "r") as file:
        content = file.read()
    # split the content by section headers (e.g., &SPECIES, &SPCGENERAL, etc.)
    sections = re.split(r"&\w+", content)
    # get all the headers by finding the section names
    section_headers = re.findall(r"&(\w+)", content)
    # remove comments
    sections = [
        section.strip() for section in sections if section[0] != "!" and section.strip()
    ]
    for header, section in zip(section_headers, sections):
        section_dict = {}
        params = re.findall(r"(\w+)\s*=\s*([-\d\.e\w]+)", section)
        for param, value in params:
            try:
                section_dict[param] = (
                    float(value) if "e" in value or "." in value else int(value)
                )
            except ValueError:
                section_dict[param] = value.strip()
        while header in parsed_data:
            header = f"{header}0"
        parsed_data[header] = section_dict

    return parsed_data

In [48]:
def preprocess(filename, spatial_ifft=False):
    dir_in = f"{ROOT}/raw/{filename}"
    dir_out = f"{ROOT}/preprocessed"
    if not os.path.exists(dir_out):
        os.makedirs(dir_out)

    ks = K_files(dir_in)
    potens, ts_slices = poten_files(dir_in)
    # get timestamps
    ts = []
    for k in ks:
        # load corresponding timestep
        with open(f"{dir_in}/{k}.dat", "r") as file:
            for line in file:
                line_split = line.split("=")
                if line_split[0].strip() == "TIME":
                    time = float(line_split[1].strip().strip(",").strip())
                    ts.append(time)
    timesteps = np.array(ts)

    # read helper vars
    time = np.loadtxt(f"{dir_in}/time.dat")
    sgrid = np.loadtxt(f"{dir_in}/sgrid")
    xphi = np.loadtxt(f"{dir_in}/xphi")
    krho = np.loadtxt(f"{dir_in}/krho")
    vpgr = np.loadtxt(f"{dir_in}/vpgr.dat")
    # number of parallel direction grid points
    ns = sgrid.shape[1] if len(sgrid.shape) > 1 else sgrid.shape[0]
    # number of x, y grid points (in real space)
    nx, ny = xphi.shape[1], xphi.shape[0]
    # number of modes in x and y direction
    nkx, nky = krho.shape[1], krho.shape[0]
    # get velocity space resolutions
    nvpar, nmu = vpgr.shape[1], vpgr.shape[0]
    
    resolution = (nvpar, nmu, ns, nkx, nky)

    # load fluxes
    fluxes = np.loadtxt(f"{dir_in}/fluxes.dat")[:, 1]
    fluxes = fluxes[ts_slices]

    # load parameters
    config = parse_input_dat(f"{dir_in}/input.dat")
    ion_temp_grad = config["SPECIES"]["rlt"]

    # create h5 file with timestamps and field data
    ifft_tag = "_ifft" if spatial_ifft else ""
    h5_filename = f"{dir_out}/{filename}{ifft_tag}.h5"
    
    stats = {"mean": [], "var": [], "min": [], "max": []}
    
    with h5py.File(h5_filename, "w") as file:
        # group for metadata (e.g. timesteps)
        metadata_group = file.create_group("metadata")
        metadata_group.create_dataset("timesteps", data=timesteps)
        metadata_group.create_dataset("fluxes", data=fluxes)
        metadata_group.create_dataset("resolution", data=resolution)
        metadata_group.create_dataset("ion_temp_grad", data=ion_temp_grad, shape=(1,))

        # group for our 6D field data
        data_group = file.create_group("data")
        for idx, (k, pot) in tqdm(
            enumerate(zip(ks, potens)), f"Processing {filename}", total=len(ks)
        ):

            # Load the full distribution function data
            with open(f"{dir_in}/{k}", "rb") as fid:
                ff = np.fromfile(fid, dtype=np.float64)

            # Reshape the distribution function
            knth = np.reshape(ff, (2, *resolution), order="F").astype("float32")
            if spatial_ifft:
                # invert fft on spatial
                knth = np.moveaxis(knth, 0, -1)
                knth = knth.copy().view(dtype=np.complex64)
                knth = np.fft.ifftn(knth, axes=(3, 4))
                knth = np.stack([knth.real, knth.imag]).squeeze()
            
            # update running averages
            k_mean = knth.mean((1, 2, 3, 4, 5))
            k_std = knth.std((1, 2, 3, 4, 5))
            k_min = knth.min((1, 2, 3, 4, 5))
            k_max = knth.max((1, 2, 3, 4, 5))
            
            stats["mean"].append(k_mean)
            stats["var"].append(k_std ** 2)
            stats["min"].append(k_min)
            stats["max"].append(k_max)

            # Add the reshaped data as a dataset to the "data" group
            k_name = "timestep_" + str(idx).zfill(5)
            data_group.create_dataset(k_name, data=knth)

            # load the potential field
            a = np.loadtxt(f"{dir_in}/{pot}")
            phi = np.reshape(a, (nx, ns, ny), order="F").astype("float32")
            poten_name = "poten_" + str(idx).zfill(5)
            data_group.create_dataset(poten_name, data=phi)
        
        stats = {k: np.stack(v) for k, v in stats.items()}
        metadata_group.create_dataset("k_mean", data=stats["mean"].mean(0))
        metadata_group.create_dataset("k_std", data=np.sqrt(stats["var"].mean(0)))
        metadata_group.create_dataset("k_min", data=stats["min"].mean(0))
        metadata_group.create_dataset("k_max", data=stats["max"].mean(0))

        return h5_filename

In [ ]:
IFFT = True
datasets = [
    "cyclone4_2_2",
    "cyclone5_2",
    "cyclone6_2",
    "cyclone7_2",
    "cyclone8_2",
    "cyclone9_2",
    "cyclone10_2",
    "cyclone11_2",
    "cyclone12_2",
    "cyclone13_2",
]

for f in datasets:
    h5_filename = preprocess(f, spatial_ifft=IFFT)
    # set rwx permissions
    try:
        os.chmod(h5_filename, 0o777)
    except PermissionError:
        pass
    # read in the structure and example field of the created h5 file
    with h5py.File(h5_filename, "r") as h5f:
        # Read the 'metadata/timesteps' dataset
        timesteps = len(h5f["data"])
        rlt = h5f["metadata/ion_temp_grad"][:]
        rlt = h5f["metadata/ion_temp_grad"][:]
        timestep_0 = h5f["data/timestep_00000"][:]
        mean, std = h5f["metadata/k_mean"][0], h5f["metadata/k_std"][0]
        min_, max_ = h5f["metadata/k_min"][0], h5f["metadata/k_max"][0]
        print(
            f"{h5_filename}:\n "
            f"\tpoints: {timesteps}, shape of timestep_00000: {timestep_0.shape}\n"
            f"\trange: ({min_:.3E}, {max_:.3E}), mean:{mean:.3E}, std:{std:.3E} \n"
            f"\trlt: {rlt}\n"
        )

Processing cyclone4_2_2: 100%|██████████| 166/166 [08:36<00:00,  3.11s/it]


/restricteddata/ukaea/gyrokinetics/preprocessed/cyclone4_2_2_ifft.h5:
 	points: 332, shape of timestep_00000: (2, 32, 8, 16, 255, 32)
	range: (-3.272E-03, 3.296E-03), mean:-3.891E-10, std:9.746E-05 
	rlt: [6.9]



Processing cyclone5_2: 100%|██████████| 166/166 [05:38<00:00,  2.04s/it]


/restricteddata/ukaea/gyrokinetics/preprocessed/cyclone5_2_ifft.h5:
 	points: 332, shape of timestep_00000: (2, 32, 8, 16, 255, 32)
	range: (-1.620E-03, 1.627E-03), mean:-6.909E-11, std:5.408E-05 
	rlt: [4.9]



Processing cyclone6_2: 100%|██████████| 166/166 [06:27<00:00,  2.34s/it]


/restricteddata/ukaea/gyrokinetics/preprocessed/cyclone6_2_ifft.h5:
 	points: 332, shape of timestep_00000: (2, 32, 8, 16, 255, 32)
	range: (-4.707E-03, 4.726E-03), mean:5.472E-10, std:1.392E-04 
	rlt: [8.9]



Processing cyclone7_2:  38%|███▊      | 63/166 [02:39<06:39,  3.87s/it]